# DocFinder — Indexeur Colab (GPU)

Ce notebook tourne sur Colab GPU et prend en charge tout le travail lourd d'embedding.

## Architecture
```
Mac (vieux) ─── /chunks NDJSON ──► Colab GPU ─── embeddings ──► Mac /admin/upsert ─── Qdrant local
            (extraction texte)     (sentence-transformers)     (FastAPI port 8000)    (port 6333)
```

Colab ne se connecte **jamais** directement à Qdrant — tout passe par le serveur FastAPI du Mac, exposé via Cloudflare Tunnel.

## Flux automatique
1. Colab poll `/admin/status` toutes les 5s via l'URL publique Cloudflare
2. Quand `status=running` → fetch `/chunks` en streaming
3. Calcule les embeddings dense sur GPU
4. POST les points vers `/admin/upsert` sur le Mac → le Mac écrit dans Qdrant local

## Prérequis
- Qdrant binaire en cours d'exécution sur le Mac (port 6333)
- Serveur FastAPI en cours d'exécution sur le Mac (port 8000 — `uvicorn server.main:app`)
- **Cloudflare Tunnel** actif : `cloudflared tunnel run --token …` (ou `./start.sh` qui le fait pour vous)
- Collection Qdrant initialisée (`python setup_qdrant.py`)


## 1. Installation des dépendances

YAKE + sparse vectors sont maintenant calculés côté Colab (pas sur le Mac) — décision D15. Gros gain CPU sur la machine locale.

In [ ]:
!pip install -q sentence-transformers requests yake

## 2. Configuration

Collez l'URL publique Cloudflare Tunnel affichée dans l'admin DocFinder (`http://localhost:8000/admin` → section Tunnel). Elle est stable entre les redémarrages — plus besoin de la recopier à chaque session.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# URL publique du serveur DocFinder (exposé via Cloudflare Tunnel)
# Stable tant que le tunnel nommé n'est pas détruit sur le dashboard Cloudflare.
# ─────────────────────────────────────────────────────────────────────────────
DOCFINDER_URL = "https://docfinder.example.com"   # ← remplacez par votre hostname

# Nom de la collection Qdrant
COLLECTION_NAME = "docfinder"

# Modèle d'embedding (identique au serveur local)
MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"

# Batch GPU : T4 tient largement 256 avec ce modèle (gain ~1,6× vs 128)
BATCH_SIZE = 256

# Safety net : flush forcé si on dépasse ce volume (plus petit = moins à perdre).
# La règle principale est toutefois un flush à la fin de chaque document (D15).
UPSERT_EVERY = 64

# Retry upsert — exponentiel (s) : 2, 4, 8
UPSERT_RETRIES = 3
UPSERT_BACKOFF = 2

# Checkpoint disque — les pending_points sont picklés avant chaque flush.
# Au redémarrage, on rejoue le checkpoint si présent (survie aux kill/crash Colab).
CHECKPOINT_PATH = "/content/docfinder_checkpoint.pkl"

# Intervalle de polling (secondes)
POLL_INTERVAL = 5

print(f"DocFinder : {DOCFINDER_URL}")
print(f"Batch GPU : {BATCH_SIZE} | Flush safety-net : {UPSERT_EVERY} | Checkpoint : {CHECKPOINT_PATH}")

## 3. Chargement du modèle sur GPU

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

model = SentenceTransformer(MODEL_NAME, device=device)
print(f"Modèle '{MODEL_NAME}' chargé sur {device}")

## 4. Test de connexion au Mac

In [ ]:
import requests

r = requests.get(f"{DOCFINDER_URL}/health", timeout=10)
r.raise_for_status()
data = r.json()
print(f"Serveur DocFinder : {data}")
assert data.get("status") == "ok", "Le serveur ne répond pas correctement"
print("Connexion OK ✓")


## 5. Moteur d'indexation

Flux : le Mac envoie du **texte brut** (NDJSON), Colab calcule YAKE + sparse + dense puis POST les points finalisés au Mac.

**Garanties de fiabilité** (D15) :
- **Flush à la fin de chaque document** → aucun doc partiellement en Qdrant
- **Checkpoint disque** picklé avant chaque flush → survit aux crashes Colab
- **Retry 3× avec backoff** sur les upserts → robuste aux micro-coupures Cloudflare
- **Reprise doc-level** via `/admin/indexed-doc-ids`

In [ ]:
import hashlib
import json
import os
import pickle
import queue
import threading
import time as _time
import requests
import yake

QUEUE_MAXSIZE = 4  # batches en avance max (évite OOM)

# YAKE — même config que le serveur local (DECISIONS D3)
_yake = yake.KeywordExtractor(lan="fr", n=3, dedupLim=0.7, top=20)


def _kw_index(kw: str) -> int:
    """MD5 → indice sparse [0, 2^20). Identique à server/indexer.py + chunks.py."""
    return int(hashlib.md5(kw.lower().encode()).hexdigest(), 16) % (2 ** 20)


def _build_sparse(text: str):
    """Vecteur sparse YAKE : (indices, values) normalisés dans [0, 1]."""
    pairs = _yake.extract_keywords(text)
    if not pairs:
        return [], []
    raw = {_kw_index(kw): 1.0 / (s + 1e-9) for kw, s in pairs}
    max_v = max(raw.values())
    return list(raw.keys()), [v / max_v for v in raw.values()]


def get_status() -> dict:
    r = requests.get(f"{DOCFINDER_URL}/admin/ping", timeout=30)
    return r.json()


def send_heartbeat() -> None:
    try:
        requests.post(
            f"{DOCFINDER_URL}/admin/colab/heartbeat",
            json={"device": device},
            timeout=5,
        )
    except Exception:
        pass


def check_paused() -> bool:
    try:
        r = requests.get(f"{DOCFINDER_URL}/admin/colab/control", timeout=5)
        return r.json().get("paused", False)
    except Exception:
        return False


def fetch_indexed_doc_ids() -> set:
    try:
        r = requests.get(f"{DOCFINDER_URL}/admin/indexed-doc-ids", timeout=30)
        ids = set(r.json().get("doc_ids", []))
        if ids:
            print(f"  Reprise : {len(ids)} docs déjà indexés — ignorés.")
        return ids
    except Exception as e:
        print(f"  Avertissement : impossible de récupérer les doc_ids existants ({e})")
        return set()


def _stream_producer(path, batch_q: "queue.Queue", skip_doc_ids: set) -> None:
    """
    Thread producteur : consomme le flux NDJSON du Mac et pousse des batches GPU.
    Le flux ne contient plus YAKE/sparse — Colab les calculera (D15).
    Un batch GPU ne mélange jamais deux documents : on flush à la fin de chaque
    document pour garantir l'atomicité au niveau doc_id.
    """
    url = f"{DOCFINDER_URL}/chunks"
    if path:
        url += f"?path={requests.utils.quote(path)}"
    batch = []
    current_doc = None
    skipped_docs = 0

    def flush_batch():
        nonlocal batch
        if batch:
            batch_q.put(batch)
            batch = []

    try:
        with requests.get(url, stream=True, timeout=300) as resp:
            resp.raise_for_status()
            for raw_line in resp.iter_lines():
                if not raw_line:
                    continue
                line = json.loads(raw_line)
                t = line.get("type")
                if t == "meta":
                    print(f"  {line['total_files']} fichiers à traiter")
                elif t == "file":
                    print(f"  {line['path']} → {line['chunks']} chunks")
                elif t == "skip":
                    print(f"  (ignoré) {line['path']}")
                elif t == "chunk":
                    doc_id = line.get("doc_id")
                    if doc_id in skip_doc_ids:
                        skipped_docs += 1
                        continue
                    # Frontière de document → flush + marqueur end-of-doc
                    if current_doc is not None and doc_id != current_doc:
                        flush_batch()
                        batch_q.put({"__end_of_doc__": current_doc})
                    current_doc = doc_id
                    batch.append(line)
                    if len(batch) >= BATCH_SIZE:
                        flush_batch()
                elif t == "done":
                    flush_batch()
                    if current_doc is not None:
                        batch_q.put({"__end_of_doc__": current_doc})
                    msg = "  Flux terminé."
                    if skipped_docs:
                        msg = f"  Flux terminé. ({skipped_docs} chunks ignorés — déjà indexés)"
                    print(msg)
                elif t == "error":
                    print(f"  ERREUR flux : {line.get('error')}")
                    break
    except Exception as e:
        print(f"  ERREUR producteur : {e}")
    finally:
        batch_q.put(None)  # sentinel de fin


def _upsert_batch(points_data: list) -> None:
    """POST vers /admin/upsert avec retry exponentiel (D15)."""
    last_err = None
    for attempt in range(UPSERT_RETRIES):
        try:
            r = requests.post(
                f"{DOCFINDER_URL}/admin/upsert",
                json=points_data,
                timeout=180,
            )
            r.raise_for_status()
            resp = r.json()
            if "error" in resp:
                raise RuntimeError(f"Upsert échoué côté Mac : {resp['error']}")
            return
        except Exception as e:
            last_err = e
            if attempt < UPSERT_RETRIES - 1:
                sleep = UPSERT_BACKOFF * (2 ** attempt)
                print(f"  ⚠ upsert tentative {attempt + 1}/{UPSERT_RETRIES} échouée : {e} — retry dans {sleep}s")
                _time.sleep(sleep)
    raise RuntimeError(f"Upsert définitivement échoué après {UPSERT_RETRIES} tentatives : {last_err}")


def _save_checkpoint(points: list) -> None:
    """Pickle les pending_points sur disque Colab pour survie aux crashes."""
    if not points:
        return
    try:
        tmp = CHECKPOINT_PATH + ".tmp"
        with open(tmp, "wb") as f:
            pickle.dump(points, f)
        os.replace(tmp, CHECKPOINT_PATH)
    except Exception as e:
        print(f"  ⚠ checkpoint échoué : {e}")


def _clear_checkpoint() -> None:
    try:
        if os.path.exists(CHECKPOINT_PATH):
            os.remove(CHECKPOINT_PATH)
    except Exception:
        pass


def _load_checkpoint() -> list:
    """Charge un checkpoint précédent (si le kernel a crashé mid-flight)."""
    if not os.path.exists(CHECKPOINT_PATH):
        return []
    try:
        with open(CHECKPOINT_PATH, "rb") as f:
            pts = pickle.load(f)
        print(f"  ↻ Checkpoint précédent trouvé : {len(pts)} points à rejouer.")
        return pts
    except Exception as e:
        print(f"  ⚠ checkpoint illisible : {e}")
        return []


def _flush(pending_points: list, label: str) -> int:
    """Flush transactionnel : checkpoint → upsert → clear checkpoint."""
    if not pending_points:
        return 0
    _save_checkpoint(pending_points)
    _upsert_batch(pending_points)
    n = len(pending_points)
    _clear_checkpoint()
    print(f"  ✓ {label} {n} points → Qdrant")
    return n


def _encode_and_enqueue(batch: list, pending_points: list) -> None:
    """Encode un batch (dense) + calcule YAKE/sparse + ajoute à pending_points."""
    texts = [c["content"] for c in batch]
    embeddings = model.encode(
        texts,
        batch_size=len(texts),
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    for chunk, emb in zip(batch, embeddings):
        sp_i, sp_v = _build_sparse(chunk["content"])
        kw_pairs = _yake.extract_keywords(chunk["content"])
        keywords = [kw for kw, _ in kw_pairs]
        p = {
            "id":      chunk["point_id"],
            "dense":   emb.tolist(),
            "payload": {
                "doc_id":      chunk["doc_id"],
                "chunk_id":    chunk["chunk_id"],
                "title":       chunk["title"],
                "path":        chunk["path"],
                "abs_path":    chunk.get("abs_path", ""),
                "doc_type":    chunk["doc_type"],
                "content":     chunk["content"],
                "keywords":    keywords,
                "chunk_index": chunk["chunk_index"],
            },
        }
        if sp_i:
            p["sparse_indices"] = sp_i
            p["sparse_values"]  = sp_v
        pending_points.append(p)
    del embeddings


def index_pipeline(path=None) -> int:
    """
    Pipeline : producteur réseau (Mac) + consommateur GPU+CPU (Colab).
    Flush systématique à la fin de chaque document (marqueur __end_of_doc__)
    + flush safety-net tous les UPSERT_EVERY points.
    """
    # Rejouer un éventuel checkpoint orphelin avant de streamer
    leftover = _load_checkpoint()
    if leftover:
        try:
            _upsert_batch(leftover)
            print(f"  ✓ Checkpoint rejoué ({len(leftover)} points)")
            _clear_checkpoint()
        except Exception as e:
            print(f"  ⚠ Rejouage du checkpoint échoué : {e}")

    skip_doc_ids = fetch_indexed_doc_ids()

    batch_q: "queue.Queue" = queue.Queue(maxsize=QUEUE_MAXSIZE)
    producer = threading.Thread(
        target=_stream_producer, args=(path, batch_q, skip_doc_ids), daemon=True
    )
    producer.start()

    total_inserted = 0
    pending_points: list = []

    while True:
        item = batch_q.get()
        if item is None:  # fin du flux
            break

        # ── Pause (Mac peut demander une pause via /admin/colab/pause) ─────
        while check_paused():
            print(f"  ⏸ En pause — en attente de reprise…", end="\r")
            _time.sleep(3)
        # ───────────────────────────────────────────────────────────────────

        # Marqueur de fin de document → flush atomique
        if isinstance(item, dict) and "__end_of_doc__" in item:
            total_inserted += _flush(pending_points, "Flush fin-de-doc")
            pending_points = []
            torch.cuda.empty_cache()
            continue

        _encode_and_enqueue(item, pending_points)

        # Safety net : ne jamais accumuler trop sans flush
        if len(pending_points) >= UPSERT_EVERY:
            total_inserted += _flush(pending_points, "Flush safety-net")
            pending_points = []
            torch.cuda.empty_cache()

    # Flush final du reste
    total_inserted += _flush(pending_points, "Flush final")
    torch.cuda.empty_cache()
    producer.join()
    return total_inserted


print("Moteur d'indexation prêt (YAKE+sparse calculés sur Colab, flush par doc).")

## 6. Mode daemon — polling automatique

Cette cellule tourne en boucle. Dès que vous cliquez **Lancer l'indexation** dans l'interface Mac (`http://localhost:8000/admin`), Colab détecte le job et traite automatiquement.

**Arrêt** : interrompre le kernel (bouton ■ ou Ctrl+C).

In [ ]:
import time

print("Daemon démarré — en attente d'un job sur le Mac…")
print(f"Poll toutes les {POLL_INTERVAL}s sur {DOCFINDER_URL}/admin/status")
print("Lancez une indexation depuis http://localhost:8000/admin")
print("─" * 60)

job_running = False

while True:
    try:
        # Heartbeat — informe le Mac que Colab est connecté + device utilisé
        send_heartbeat()

        status = get_status()
        s = status.get("status")

        if s == "running" and not job_running:
            job_running = True
            print(f"\n[{time.strftime('%H:%M:%S')}] Job détecté ! Démarrage du pipeline…")
            try:
                n = index_pipeline()
                print(f"\n[{time.strftime('%H:%M:%S')}] Terminé — {n} points insérés dans Qdrant.")
            except Exception as e:
                print(f"[{time.strftime('%H:%M:%S')}] ERREUR : {e}")

        elif s in ("done", "error", "idle") and job_running:
            job_running = False
            print(f"[{time.strftime('%H:%M:%S')}] Job {s}. En attente du prochain…")
            print("─" * 60)

        elif s == "running" and job_running:
            done  = status.get("done", 0)
            total = status.get("total", 0)
            pct   = status.get("progress_pct", 0)
            print(f"  Mac : {done}/{total} fichiers ({pct}%)…", end="\r")

    except requests.exceptions.ConnectionError:
        print(f"[{time.strftime('%H:%M:%S')}] Mac inaccessible, nouvelle tentative…")
    except Exception as e:
        print(f"[{time.strftime('%H:%M:%S')}] Erreur réseau : {e}")

    time.sleep(POLL_INTERVAL)


## 7. Indexation manuelle (optionnel)

Si vous ne voulez pas le daemon, lancez une indexation unique ici.

In [ ]:
# Chemin iCloud sur le Mac (optionnel — utilise le défaut si vide)
CUSTOM_PATH = ""  # ex: "/Users/julien/Library/Mobile Documents/com~apple~CloudDocs"

print("Démarrage du pipeline…")
n = index_pipeline(CUSTOM_PATH or None)
print(f"\nTerminé — {n} points insérés.")


## 8. Vérification — test de l'endpoint upsert

In [ ]:
# Vérifie que le Mac est joignable et que le moteur de recherche est prêt
r = requests.get(f"{DOCFINDER_URL}/health", timeout=10)
r.raise_for_status()
health = r.json()
print(f"Santé serveur : {health}")

# Vérifie que /admin/status répond
r2 = requests.get(f"{DOCFINDER_URL}/admin/status", timeout=10)
r2.raise_for_status()
status = r2.json()
print(f"Statut job   : {status['status']} — {status['done']}/{status['total']} fichiers, {status['chunks']} chunks")
